In [1]:
from glob import glob
import pandas as pd
import openpyxl
import tqdm
from itertools import chain
from util import search_location, extract_urls


In [14]:
root_path = "../data/schede mappatura/"

skip = {"david_ruth_FEGB_E_00007": {"rows": 6, "cols": 1}}

In [3]:
chrono_schede = glob(f"{root_path}*/chronotop*")
chrono_schede

['../data/schede mappatura/bruenn_jehoshua_IS_S_00028/chronotopoi_bruenn_jehoshua_bozza.xlsx',
 '../data/schede mappatura/david_ruth_FEGB_E_00007/chronotopi_ruth_david_FEGB_E_00007.xlsx',
 '../data/schede mappatura/bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx',
 '../data/schede mappatura/bruenn_ charlotte_IS_S_00027/chronotopoi_charlotte_bruenn_IS_S_00027 (bozza).xlsx',
 '../data/schede mappatura/stern_IS_S_00142/chronotopi_josef_stern_IS_S_00142.xlsx']

In [15]:
current = "bruenn_jehoshua_IS_S_00028/chronotopoi_bruenn_jehoshua_bozza.xlsx"
# current = "david_ruth_FEGB_E_00007/chronotopi_ruth_david_FEGB_E_00007.xlsx"
current = "bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx"
current = "stern_IS_S_00142/chronotopi_josef_stern_IS_S_00142.xlsx"

In [16]:
overview = {'bruenn_jehoshua_IS_S_00028/chronotopoi_bruenn_jehoshua_bozza.xlsx': [
    'Foglio1',
],
 # 'david_ruth_FEGB_E_00007/chronotopi_ruth_david_FEGB_E_00007.xlsx': ['Ruth L.David',
 #  'Mutter',
 #  'Vater',
 #  'Isaak Oppenheimer',
 #  'Ernst',
 #  'Werner',
 #  'Hanna',
 #  'Michael',
 #  'Fedora',
 #  'Liese',
 #  'Herbert Aron David',
 #  'Alice Urbach'],
 'bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx': ['Charlotte Bruenn geb. Hirsch',
  'Ludwig Hirsch',
  'Gutta Hirsch geb. Dingfelder',
  'Alice Hirsch (Estryn)',
  'Erich Hirsch',
  'Walter Hirsch',
  'Alfons Justin Hirsch',
  'Margot Hirsch',
  ],
 'stern_IS_S_00142/chronotopi_josef_stern_IS_S_00142.xlsx': ['Josef Stern',
  'Julius Stern',
  'Esther Stern',
  'Sonja Stern',
  'Großvater M David Kaminka',
  'Großmutter M Betty Kaminka geb.',
  'Großvater V Isaak Stern',
  'Großmutter V Auguste Stern geb ',
  'Mutter 2 Claire Stern']}

In [17]:
first = True
df_list = []
for k,v  in overview.items():
  for s in v:
    name = k.split("/")[0]
    # print(k,s)
    if name in skip:  
        df = pd.read_excel(root_path + k, sheet_name=s, skiprows=skip[name]["rows"]).iloc[:, skip[name]["cols"]:]
    else:
        df = pd.read_excel(root_path + k, sheet_name=s) 

    print(k, s, df.columns)
    df.columns = ["event", "location", "timespan", "notes", "links"]

    # merge columns 6+ to notes
    df["notes"] = df[["notes"] + list(df.columns[6:])].apply(
        lambda row: ' '.join(row.dropna().astype(str)), axis=1
    )
    df = df.drop(df.columns[6:], axis=1)

    df["protagonist"] = name
    df["name"] = s
    df_list += [df]
df = pd.concat(df_list, axis=0)
df.fillna("", inplace=True)
df

bruenn_jehoshua_IS_S_00028/chronotopoi_bruenn_jehoshua_bozza.xlsx Foglio1 Index(['EVENT', 'LOCATION', 'TIMESPAN', 'NOTES', 'LINKS'], dtype='str')
bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx Charlotte Bruenn geb. Hirsch Index(['Event', 'Location', 'Timespan', 'Notes', 'Links'], dtype='str')
bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx Ludwig Hirsch Index(['Event', 'Location', 'Timespan', 'Notes', 'Links'], dtype='str')
bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx Gutta Hirsch geb. Dingfelder Index(['Event', 'Location', 'Timespan', 'Notes', 'Links'], dtype='str')
bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx Alice Hirsch (Estryn) Index(['Event', 'Location', 'Timespan', 'Notes', 'Links'], dtype='str')
bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx Erich Hirsch

,event,location,timespan,notes,links,protagonist,name
0,Lebenslauf - Geburt / Alter Heimat,\t\nAllenstein (Ostpreußen)https://www.geoname...,19. Januar 1913,,,bruenn_jehoshua_IS_S_00028,Foglio1
1,Lebenslauf - Tod des Vaters,Allenstein,1923,24min01s. - 24min04s.,,bruenn_jehoshua_IS_S_00028,Foglio1
2,Lebenslauf - Schuljahren (Realschule),Allenstein,ca. 1927/1928,"Zuerst christliche Volksschule, dann (von der ...",,bruenn_jehoshua_IS_S_00028,Foglio1
3,Lebenslauf - Kaufmännische Lehre,Allenstein,1931,,,bruenn_jehoshua_IS_S_00028,Foglio1
4,Lebenslauf - Militärversuch,Allenstein,1931,Er hat 3 Jahren bei Frankenstein & Co (Warenha...,,bruenn_jehoshua_IS_S_00028,Foglio1
...,...,...,...,...,...,...,...
1,Deportation,Gießen https://www.wikidata.org/wiki/Q7904 Wal...,16.9.1942\n,,,stern_IS_S_00142,Mutter 2 Claire Stern
2,Transport (Abfahrt),Darmstadt https://www.wikidata.org/wiki/Q2973 ...,1942-09-30 00:00:00,,https://memoryoftreblinka.org/transports-to-tr...,stern_IS_S_00142,Mutter 2 Claire Stern
3,Transport (Ankunft),Treblinka https://www.wikidata.org/wiki/Q15201...,1942-10-02 00:00:00,,,stern_IS_S_00142,Mutter 2 Claire Stern
4,Mord,Treblinka https://www.wikidata.org/wiki/Q15201...,1942,Stolperstein Gießen,https://collections.yadvashem.org/en/names/135...,stern_IS_S_00142,Mutter 2 Claire Stern


In [ ]:
# locs = {n:l for l, n in df["location"].apply(lambda x: extract_urls(x)).to_list()}
locs = {}
for row in df["location"]:
    # print(row)
    # print(extract_urls(row))
    for substring, urls in extract_urls(row):
        locs[substring] = urls
print(locs)    
# pd.DataFrame.from_dict(locs, orient="index").to_excel("locations.xlsx")
rows = [{"location": k} | v for k, v in locs.items()]
pd.DataFrame(rows).to_excel("locations.xlsx", index=False)

In [ ]:
set(df["event"])

In [ ]:
set(df["timespan"])

In [ ]:
set(df["notes"])

In [ ]:
set(df["links"])